# 02 - Verifikasi Kondisi Pencahayaan

Mengukur luminansi rata-rata per kondisi (normal/dim/low) dan membuat contact sheet
contoh citra. Tujuannya memastikan urutan luminansi **normal > dim > low**.

Keluaran: `results/summary/luminance_stats.csv`, `results/summary/lighting_samples.png`.

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Image

# Temukan root proyek (folder berisi configs/experiment.yaml), lalu jadikan cwd.
_cands = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(p for p in _cands if (p / "configs" / "experiment.yaml").exists())
os.chdir(ROOT)
for _p in (ROOT / "src", ROOT / "scripts"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from vehicle_lighting.config import load_config, resolve
cfg = load_config()
print("ROOT   :", ROOT)
print("config :", ROOT / "configs" / "experiment.yaml")
print("kelas  :", cfg["data"]["classes"])
print("model  :", cfg["models"])

## 2. Hitung luminansi per citra

In [ ]:
import cv2, numpy as np, pandas as pd
from vehicle_lighting.lighting import mean_luminance

df = pd.read_csv(resolve(cfg, cfg["data"]["manifest"]))
df["luminance"] = [
    mean_luminance(cv2.cvtColor(cv2.imread(str(resolve(cfg, p))), cv2.COLOR_BGR2RGB))
    for p in df["path"]
]
out = resolve(cfg, cfg["results_dir"]) / "summary"
out.mkdir(parents=True, exist_ok=True)
stats = df.groupby("lighting")["luminance"].describe()[["count", "mean", "std", "min", "max"]]
stats.to_csv(out / "luminance_stats.csv")
stats

## 3. Contact sheet (kolom: normal, dim, low)

In [ ]:
order = [l for l in ["normal", "dim", "low"] if l in set(df.lighting)]
n = int(min(4, (df.lighting == "normal").sum()))
base = df[df.lighting == "normal"].sample(n, random_state=0)
rows = []
for r in base.itertuples():
    tiles = []
    for lt in order:
        m = df[(df.lighting == lt) & (df.group_id == r.group_id)]
        p = m.iloc[0].path if len(m) else r.path
        tiles.append(cv2.resize(cv2.imread(str(resolve(cfg, p))), (224, 224)))
    rows.append(np.hstack(tiles))
sheet = np.vstack(rows)
cv2.imwrite(str(out / "lighting_samples.png"), sheet)
Image(filename=str(out / "lighting_samples.png"))

**Gate tahap 2:** `mean` luminansi `normal > dim > low`. Lanjut ke `03_pelatihan.ipynb`.